# LOAD DATA AND SELECT MOST RECENT SUBMISSIONS

In [0]:


trusts= spark.read.table("udal_gold_restricted.ukhd_ods.Provider_Site")

trusts.createOrReplaceTempView("trusts")

In [0]:
bbcs_maternity_trusts = ["RNA",  "RXK", "RBK", "RL4", "RRK", "RQ3"] 

In [0]:

# Header data for deduplicating
msds_header = spark.read.table("udal_silver_restricted.mesh_msds.MSD000Header")

msds_header.createOrReplaceTempView("msds_header")

# Booking data
booking = spark.read.table("udal_silver_restricted.mesh_msds.MSD101PregnancyBooking")

booking.createOrReplaceTempView("booking")

# Contact data
contact = spark.read.table("udal_silver_restricted.mesh_msds.MSD201CareContactPreg")

contact.createOrReplaceTempView("contact")

# Birth spells data
spell = spark.read.table("udal_silver_restricted.mesh_msds.MSD501HospProvSpell")

# Labour data
labour = spark.read.table("udal_silver_restricted.mesh_msds.MSD301LabourDelivery")

labour.createOrReplaceTempView("labour")

# Baby data
baby = spark.read.table("udal_silver_restricted.mesh_msds.MSD401BabyDemographics")

baby.createOrReplaceTempView("baby")

# Mother demographics
mother = spark.read.table("udal_silver_restricted.mesh_msds.MSD001MotherDemog")

mother.createOrReplaceTempView("mother")

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW msds_latest_submission AS 
SELECT
    UniqSubmissionID, 
    auditid

FROM (
    SELECT 
        UniqSubmissionID, 
        OrgIDProvider,
        RPStartDate, 
        auditid,
        ROW_NUMBER() OVER (PARTITION BY RPStartDate, OrgIDProvider ORDER BY UploadDate DESC) AS rn
    FROM msds_header
    WHERE RPEndDate<'2026-08-31'
    ) AS Sub

WHERE rn = 1

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW msds_labour_valid AS 

SELECT *,
ROW_NUMBER() OVER (ORDER BY (SELECT 1)) AS row_num
FROM labour AS labour
WHERE EXISTS
( -- just grabs patients where the UniqueSubmissionID relates to the 'latest' submission
  SELECT  UniqSubmissionID -- select just the latest submissions
  FROM msds_latest_submission AS Latest
  WHERE labour.UniqSubmissionID = Latest.UniqSubmissionID
) 

In [0]:
%sql

CREATE OR REPLACE TEMPORARY VIEW msds_baby_valid AS 

SELECT *, 
       ROW_NUMBER() OVER (ORDER BY (SELECT 1)) AS row_num
FROM baby AS baby
WHERE EXISTS (
  SELECT 1
  FROM msds_latest_submission AS Latest
  WHERE baby.UniqSubmissionID = Latest.UniqSubmissionID
)



In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW msds_booking_valid AS 

SELECT ROW_NUMBER() OVER (ORDER BY (SELECT 1)) AS row_num, *
FROM booking AS booking
WHERE EXISTS
( -- just grabs patients where the UniqueSubmissionID relates to the 'latest' submission
  SELECT  UniqSubmissionID -- select just the latest submissions
  FROM msds_latest_submission AS Latest
  WHERE booking.UniqSubmissionID = Latest.UniqSubmissionID
)

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW msds_contact_valid AS 

SELECT ROW_NUMBER() OVER (ORDER BY (SELECT 1)) AS row_num, *
FROM contact AS contact
WHERE EXISTS
( -- just grabs patients where the UniqueSubmissionID relates to the 'latest' submission
  SELECT  UniqSubmissionID -- select just the latest submissions
  FROM msds_latest_submission AS Latest
  WHERE contact.UniqSubmissionID = Latest.UniqSubmissionID
)

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW msds_mother_valid AS 

SELECT ROW_NUMBER() OVER (ORDER BY (SELECT 1)) AS row_num, *
FROM mother AS mother
WHERE EXISTS
( -- just grabs patients where the UniqueSubmissionID relates to the 'latest' submission
  SELECT  UniqSubmissionID -- select just the latest submissions
  FROM msds_latest_submission AS Latest
  WHERE mother.UniqSubmissionID = Latest.UniqSubmissionID
)

# SELECT DATA FROM SOURCE TABLES

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW births_2326_baby_data AS 

SELECT DISTINCT
  a.Person_ID_Mother,
  a.UniqPregID,
  a.AgeAtBirthMother,
  CASE 
  WHEN (a.YearOfBirthBaby LIKE '2025' AND a.MonthOfBirthBaby IN ('04', '05','06', '07', '08', '09', '10', '11', '12')) OR
 (a.YearOfBirthBaby LIKE '2026' AND a.MonthOfBirthBaby IN ('01', '02','03')) THEN '2025/26' 
   WHEN (a.YearOfBirthBaby LIKE '2024' AND a.MonthOfBirthBaby IN ('04', '05','06', '07', '08', '09', '10', '11', '12')) OR
 (a.YearOfBirthBaby LIKE '2025' AND a.MonthOfBirthBaby IN ('01', '02','03')) THEN '2024/25' 
    WHEN (a.YearOfBirthBaby LIKE '2023' AND a.MonthOfBirthBaby IN ('04', '05','06', '07', '08', '09', '10', '11', '12')) OR
 (a.YearOfBirthBaby LIKE '2024' AND a.MonthOfBirthBaby IN ('01', '02','03')) THEN '2023/24' 
  ELSE null END AS birth_yr,
  CONCAT(a.YearOfBirthBaby, '-', a.MonthOfBirthBaby) AS birth_date,
  b.Trust_Code2 as birth_provider,
  CASE WHEN (OrgSiteIDActualDelivery LIKE 'ZZ201' OR SettingPlaceBirth LIKE '04') THEN 1 ELSE 0 END AS home_birth, ---ZZ888 is non-NHS site but isn't used in data. ZZ777 is in transit and only occurs twice.
 no_of_babies
 
    FROM msds_baby_valid a

  LEFT JOIN 
  (SELECT *,
    CASE 
      WHEN Trust_Code LIKE 'RR1' THEN 'RRK'
      WHEN Trust_Code LIKE 'RW6' THEN 'RM3'
      WHEN Trust_Code LIKE 'RAP' THEN 'RAL'
      ELSE Trust_Code 
    END AS Trust_Code2  ---Correcting 3 old codes that are causing duplicates
  FROM trusts) b

  ON a.OrgSiteIDActualDelivery=b.Site_Code

  LEFT JOIN
  (SELECT 
  Person_ID_Mother,
  UniqPregID,
  MAX(BirthsPerLabandDel) AS no_of_babies
  FROM msds_labour_valid

  GROUP BY
  Person_ID_Mother,
  UniqPregID
  ) c
  ON
 a.Person_ID_Mother = c.Person_ID_Mother AND
  a.UniqPregID =c.UniqPregID

  WHERE a.Der_Financial_Year IN ('2022/23','2023/24','2024/25', '2025/26', '2026/27')
    AND a.BirthOrderMaternitySUS ='1'   --- if multiples only select first baby record
    AND a.Person_ID_Mother IS NOT NULL
    AND a.UniqPregID IS NOT NULL

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW booking_appts AS 

SELECT DISTINCT
a.EDDAgreed, --- Estimated due date
a.AntenatalAppDate,
a.OrgSiteIDBooking AS booking_site,
a.Person_ID_Mother,
a.UniqPregID,
a.AgeAtBookingMother,
b.Trust_Code AS booking_provider,
c.OrgSiteIDOfTreat AS contact_site,
(CASE 
WHEN a.EDDAgreed BETWEEN '2025-04-01' AND '2026-03-31' THEN '2025/26' 
WHEN a.EDDAgreed BETWEEN '2024-04-01' AND '2025-03-31' THEN '2024/25' 
WHEN a.EDDAgreed BETWEEN '2023-04-01' AND '2024-03-31' THEN '2023/24' 
ELSE null END) AS edd_yr

FROM (
  SELECT  *,
         DENSE_RANK() OVER (PARTITION BY Person_ID_Mother, UniqPregID 
         ORDER BY AntenatalAppDate ASC
         ) AS rn
  FROM msds_booking_valid    ---- select the first date if multiple booking appts
) a

LEFT JOIN trusts b
ON a.OrgSiteIDBooking=b.Site_Code

LEFT JOIN (SELECT
Person_ID_Mother,
UniqPregID,
CContactDate,
OrgSiteIDOfTreat

FROM msds_contact_valid

WHERE AttendCode IN ('5', '6')    --- contacts has to be attended
 ) c
ON a.Person_ID_Mother = c.Person_ID_Mother
   AND a.UniqPregID = c.UniqPregID
   AND a.AntenatalAppDate = c.CContactDate

WHERE (a.EDDAgreed > '2023-04-01' OR   ---expected to give birth in 2025/26, or later (will filter final results after join) but don't want to remove booking records that might result in prem births in 25/26
(a.EDDAgreed IS NULL AND a.AntenatalAppDate BETWEEN '2022-09-01' AND '2026-03-31'))   ----where due date is missing select records based on appt date being in the 7 prior to 25/26 or within the yr. Wide date window to avoid excluding too many too early
AND a.rn = 1  -- select only first appt date, or multiple if they have the same date
AND a.Person_ID_Mother is not NULL
AND a.UniqPregID is not NULL

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW mother_demo AS 

SELECT DISTINCT
  Person_ID_Mother,
  OrgIDICBRes,   --- ICB of residence
  LAD_UAMother
FROM (
  SELECT *,
         DENSE_RANK() OVER (PARTITION BY Person_ID_Mother 
         ORDER BY RPEndDate DESC
         ) AS rn 
  FROM msds_mother_valid
  WHERE LAD_UAMother is not null
    AND Person_ID_Mother is not null
    AND RPEndDate<'2026-06-01'
) sub    --- select the most recent ICB of residence/LA if more than 1
WHERE rn = 1

---- Fairly poor recording of LA, but the LSOA and other geography fields are no better
----https://digital.nhs.uk/data-and-information/data-collections-and-data-sets/data-sets/maternity-services-data-set/guidance/data-quality-dashboard
--- Dashboard suggests good completeness but significant number of postcodes are recorded ZZ99

# COMBINING DATA

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW combined_data_all AS 

SELECT 
 COALESCE(a.Person_ID_Mother, b.Person_ID_Mother) AS Person_ID_Mother,
COALESCE(a.UniqPregID, b.UniqPregID) AS UniqPregID,
a.birth_provider,
a.AgeAtBirthMother,  
a.birth_yr,
a.birth_date, 
a.home_birth,
a.no_of_babies,
b.edd_yr, 
b.AntenatalAppDate,
b.booking_provider,
b.AgeAtBookingMother,
c.OrgIDICBRes,
c.LAD_UAMother,
d.Trust_Code AS contact_provider
 
FROM births_2326_baby_data a

FULL JOIN booking_appts b
  ON a.Person_ID_Mother = b.Person_ID_Mother
  AND a.UniqPregID = b.UniqPregID

  LEFT JOIN mother_demo c
  ON
  COALESCE(a.Person_ID_Mother, b.Person_ID_Mother) = c.Person_ID_Mother

  LEFT JOIN trusts d
  ON contact_site=Site_Code


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW activity_of_interest AS 

  SELECT 
  Person_ID_Mother,
  UniqPregID, 
  AgeAtBirthMother,
  AgeAtBookingMother,
  AntenatalAppDate,
  edd_yr,
  birth_yr,
  birth_provider,
  birth_date,
  home_birth,
  no_of_babies,
  COALESCE(booking_provider, contact_provider) AS booking_provider_filled,
  OrgIDICBRes as resident_icb,
  LAD_UAMother as la_of_mother

  FROM combined_data_all

  WHERE (birth_yr IN ('2025/26', '2024/25', '2023/24') OR
        (edd_yr IN ('2025/26', '2024/25', '2023/24') AND birth_date IS NULL))


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW remove_duplicate_nulls_birth AS 
WITH flagged_births AS (
  SELECT *,
         -- Yields a 1 if a valid booking provider exists for this pregnancy group
         MAX(CASE WHEN birth_provider IS NOT NULL THEN 1 ELSE 0 END) 
             OVER (PARTITION BY Person_ID_Mother, UniqPregID) AS has_valid_birth
  FROM activity_of_interest
)
SELECT * 
FROM flagged_births
-- Exclude rows that have no provider filled ONLY IF a valid provider row exists somewhere else for that pregnancy
WHERE NOT (birth_provider IS NULL AND has_valid_birth = 1);


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW remove_duplicate_nulls_booking AS 
SELECT DISTINCT 
Person_ID_Mother,
  UniqPregID, 
  AgeAtBirthMother,
  AgeAtBookingMother,
  AntenatalAppDate,
  edd_yr,
  birth_yr,
  birth_provider,
  birth_date,
  home_birth,
  no_of_babies,
  booking_provider_filled,
  resident_icb,
  la_of_mother

FROM (WITH flagged_births AS (
  SELECT *,
         MAX(CASE WHEN booking_provider_filled IS NOT NULL THEN 1 ELSE 0 END) 
             OVER (PARTITION BY Person_ID_Mother, UniqPregID) AS has_valid_booking
  FROM remove_duplicate_nulls_birth
)
SELECT *
FROM flagged_births
-- Exclude rows that have no provider filled ONLY IF a valid provider row exists somewhere else for that pregnancy
WHERE NOT (booking_provider_filled IS NULL AND has_valid_booking = 1))

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW dupe_births AS 
WITH dupes AS (
SELECT *,
    COUNT(*) OVER(PARTITION BY Person_ID_Mother, UniqPregID)  AS count
FROM remove_duplicate_nulls_booking

)

SELECT *
FROM dupes
WHERE count >1


In [0]:

%sql
CREATE OR REPLACE TEMPORARY VIEW distinct_records AS 
WITH adding_row_hash AS (
SELECT *,
 sha2(concat_ws('||', 
    coalesce(cast(Person_ID_Mother as string), 'N/A'),
    coalesce(cast(UniqPregID as string), 'N/A'), 
    coalesce(cast(AgeAtBirthMother as string), 'N/A'),
    coalesce(cast(AgeAtBookingMother as string), 'N/A'),
    coalesce(cast(AntenatalAppDate as string), 'N/A'),
    coalesce(cast(edd_yr as string), 'N/A'),
    coalesce(cast(birth_yr as string), 'N/A'),
    coalesce(cast(birth_provider as string), 'N/A'),
    coalesce(cast(birth_date as string), 'N/A'),
    coalesce(cast(home_birth as string), 'N/A'),
    coalesce(cast(no_of_babies as string), 'N/A'),
    coalesce(cast(booking_provider_filled as string), 'N/A'),
    coalesce(cast(resident_icb as string), 'N/A'),
    coalesce(cast(la_of_mother as string), 'N/A')
  ), 256) AS row_hash
FROM remove_duplicate_nulls_booking
),
ranked_records AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY Person_ID_Mother, AntenatalAppDate
      ORDER BY birth_date DESC,
      row_hash DESC
    ) AS rn  
  FROM adding_row_hash
)

SELECT
  Person_ID_Mother,
  UniqPregID, 
  AgeAtBirthMother,
  AgeAtBookingMother, 
  AntenatalAppDate,
  edd_yr,
  birth_yr,
  birth_provider,
  birth_date,
  home_birth,
  no_of_babies,
  booking_provider_filled AS booking_provider,
  resident_icb,
 la_of_mother
  
 
FROM ranked_records
WHERE rn = 1 


   ---- Removing those with duplicates due to different providers or 2 different UniqPregIDs. Set so the same duplicated records are removed each time.



In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW final_cohort_2326 AS 

SELECT *,

CASE WHEN resident_icb IN ('QHL', 'QUA')  OR la_of_mother IN ('E08000025', 'E08000029', 'E08000030', 'E08000031', 'E08000027','E08000028')THEN 1 ELSE 0 END AS resident_flag,
  CASE WHEN birth_provider IN ("RNA",  "RXK", "RBK", "RL4", "RRK", "RQ3") THEN 1 ELSE 0 END AS birth_provider_flag,
  CASE WHEN booking_provider IN ("RNA",  "RXK", "RBK", "RL4", "RRK", "RQ3") THEN 1 ELSE 0 END AS booking_provider_flag,
  CASE WHEN birth_date IS NOT NULL THEN 1 ELSE 0 END AS birth_flag

  FROM distinct_records a

  WHERE (resident_icb LIKE 'QHL' OR resident_icb LIKE 'QUA'
     OR la_of_mother IN ('E08000025', 'E08000029', 'E08000030', 'E08000031', 'E08000027','E08000028')
      OR booking_provider IN ("RNA", "RXK", "RBK", "RL4", "RRK", "RQ3")
      OR birth_provider IN ("RNA", "RXK", "RBK", "RL4", "RRK", "RQ3"))

 ---- Data includes all births where booking or birth occurred in BBSC cluster area or where women were living in the cluster area. 


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW total_births_by_LA_2326 AS 
SELECT 
la_of_mother,
birth_yr,
CASE WHEN birth_yr is null THEN edd_yr ELSE null END AS estimated_yr,
AgeAtBirthMother,
AgeAtBookingMother,
COUNT(*) AS count
    
FROM distinct_records

-----WHERE birth_date IS NOT NULL---- only includes births, not antenatal starts

GROUP BY 
la_of_mother,
AgeAtBirthMother,
AgeAtBookingMother,
birth_yr,
CASE WHEN birth_yr is null THEN edd_yr ELSE null END

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW demographics AS 
SELECT DISTINCT
a.Person_ID_Mother,
STRING_AGG(DISTINCT b.LSOAMother2011, '|') AS lsoa_2011, STRING_AGG(DISTINCT CASE 
                 WHEN b.EthnicCategoryMother IS NOT NULL 
                  AND b.EthnicCategoryMother NOT IN ('Z', '99') 
                 THEN b.EthnicCategoryMother 
               END, '|') AS ethnicity

FROM final_cohort_2326 a

LEFT JOIN  (SELECT *,
         DENSE_RANK() OVER (PARTITION BY Person_ID_Mother 
         ORDER BY RPEndDate DESC
         ) AS rn 
           
           FROM msds_mother_valid 

           QUALIFY rn = 1
           )b

ON a.Person_ID_Mother=b.Person_ID_Mother

GROUP BY
a.Person_ID_Mother
       





In [0]:
#Read table into a dataframe for use in python
final_cohort_2326 = spark.table("final_cohort_2326")
total_births_by_LA_2326 = spark.table("total_births_by_LA_2326")
demographics = spark.table("demographics")

# set path to lake mart
lake = "udalstdataanalysisprod.dfs.core.windows.net"
container = "analytics-projects"
folder = "StrategyUnit/StrategyUnit/analytics_modelling/bbcs_maternity/"
eval_path = "abfss://" + container + "@" + lake + "/" + folder

# save to SU lake mart
final_cohort_2326.write.mode("overwrite").parquet(eval_path + "final_cohort_2326.parquet")
total_births_by_LA_2326.write.mode("overwrite").parquet(eval_path + "total_births_by_LA_2326.parquet")
demographics.write.mode("overwrite").parquet(eval_path + "demographics.parquet")

# CHECKS OF THE DATA 


In [0]:
df = spark.table("final_cohort_2326")
display(df.select("Person_ID_Mother").distinct().count())

In [0]:
df = spark.table("final_cohort_2326")
display(df.count())



In [0]:
%sql
WITH dupes AS (
SELECT *,
    COUNT(*) OVER(PARTITION BY Person_ID_Mother, UniqPregID)  AS count
FROM final_cohort_2326

)

SELECT *
FROM dupes
WHERE count >1



In [0]:
%sql
SELECT DISTINCT
b.*
FROM dupe_births a

LEFT JOIN 
(SELECT * 
FROM final_cohort_2326) b
 ON a.Person_ID_Mother = b.Person_ID_Mother
AND a.UniqPregID = b.UniqPregID

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW duplicate_uniqpregid AS 

WITH ranked_dupes AS (
    SELECT DISTINCT
    Person_ID_Mother,
    UniqPregID,
    AntenatalAppDate

    FROM remove_duplicate_nulls_booking
),  count_data AS(

SELECT
    Person_ID_Mother,
    UniqPregID,
    AntenatalAppDate,
    COUNT(*) OVER(PARTITION BY Person_ID_Mother, AntenatalAppDate)  AS count
FROM ranked_dupes
)

SELECT *
FROM count_data
WHERE count >1


In [0]:
%sql
select *
from duplicate_uniqpregid a

LEFT JOIN remove_duplicate_nulls_booking b
ON a.Person_ID_Mother = b.Person_ID_Mother and
a.UniqPregID =b.UniqPregID

In [0]:
%sql
---- Just checking the numbers in the data
WITH distinct_rows AS(
    SELECT 
    Person_ID_Mother,
    UniqPregID,
    MAX(EDDAgreed) AS max_edd

    FROM booking_appts
    GROUP BY Person_ID_Mother, UniqPregID
)

SELECT
COUNT(CASE WHEN max_edd IS NULL THEN 1 END) as count_null,
COUNT(CASE WHEN max_edd IS NOT NULL THEN 1 END) as count_not_null

FROM distinct_rows

----Approx 0.9% of records have missing estimated due dates

In [0]:
%sql
WITH la_count AS (
    SELECT 
        la_of_mother,
        COUNT(*) AS count
    FROM final_cohort_2326
    GROUP BY la_of_mother
)
SELECT 
    CASE 
        WHEN count > 100 THEN la_of_mother
        ELSE 'Other'
    END AS la_group,
    SUM(count) AS total_count
FROM la_count
GROUP BY 
    CASE 
        WHEN count > 100 THEN la_of_mother
        ELSE 'Other'
    END

In [0]:
%sql
select *
from  total_births_by_LA_2326
where birth_yr is null

In [0]:
%sql
SELECT *
FROM total_births_by_LA_2326
where birth_yr LIKE '2025/26' AND la_of_mother is null